# Week 3, Day 1: Advanced Training Techniques

## Motivation
In B1, two problems were observed:
- **Oscillating val loss** — the learning rate was too high for fine-grained convergence late in training
- **Unnecessary epochs** — val loss plateaued around epoch 20 but training continued for 100 epochs

This session introduces three techniques to address these problems.

---

## 1. Learning Rate Scheduling
A fixed learning rate is inefficient across the full training run:
- **Early training:** model is far from optimum — large steps are efficient
- **Late training:** model is close to optimum — large steps cause overshooting and oscillation

`ReduceLROnPlateau` monitors a metric (val loss) and reduces the learning rate by a factor when no improvement is detected for a specified number of epochs.
```python
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',       # lower val loss = better
    factor=0.5,       # halve the learning rate on plateau
    patience=5,       # epochs without improvement before reducing
    min_lr=1e-6       # never reduce below this value
)
```

---

## 2. Early Stopping
Continuing to train after val loss plateaus wastes compute and risks overfitting.
Early stopping halts training when val loss has not improved for a set number of epochs.

Three things to track:
- Best val loss seen so far
- Patience counter — epochs since last improvement
- Best model weights — saved at the point of lowest val loss
```python
# Save best weights
torch.save(model.state_dict(), 'best_model.pt')

# Restore best weights after early stopping
model.load_state_dict(torch.load('best_model.pt'))
```

**Why `state_dict()` and not the full model?**
`state_dict()` saves only the weight tensors as a dictionary — no class definition embedded.
Saving the full model object with pickle ties the file to a specific class location, making it
fragile across machines and refactors. `state_dict()` is portable and robust.

---

## 3. Cross-Validation (upcoming)
B1 results were sensitive to the random train/val/test split.
Cross-validation gives a more robust performance estimate by training and evaluating
across multiple splits of the data.

---

## Key Parameters

| Technique | Parameter | Role |
|---|---|---|
| ReduceLROnPlateau | `mode` | `"min"` for loss, `"max"` for accuracy/R² |
| ReduceLROnPlateau | `factor` | Multiplicative reduction factor (e.g. 0.5 = halve) |
| ReduceLROnPlateau | `patience` | Epochs without improvement before reducing LR |
| ReduceLROnPlateau | `cooldown` | Epochs to ignore after a LR reduction before resuming monitoring |
| ReduceLROnPlateau | `min_lr` | Lower bound on learning rate |
| Early Stopping | `patience` | Epochs without improvement before halting training |

In [3]:
'''
Full training loop for the ESOL dataset incorporating both ReduceLROnPlateau and early stopping. 
Use patience of 10 for early stopping and patience of 5 for the scheduler
'''
import pandas as pd

url = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
df = pd.read_csv(url)
compounds = df[["Compound ID", "smiles", "measured log solubility in mols per litre"]].copy()

try:
    from rdkit import Chem
except ImportError:
    print("RDKit is not installed. Please install it to proceed.")
    !pip install rdkit
    from rdkit import Chem

def smiles_to_rdkit_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: # None is returned silently for invalid SMILES
        return "Error"
    return mol

# Insert a new column "rdkit_mol" by applying the conversion function to the "smiles" column
compounds["rdkit_mol"] = compounds["smiles"].apply(smiles_to_rdkit_mol)
error_compounds = compounds[compounds["rdkit_mol"] == "Error"]
print(f"Number of compounds with invalid SMILES: {len(error_compounds)}")
valid_compounds = compounds[compounds["rdkit_mol"] != "Error"]
# Step 3: Compute descriptors for valid compounds
from rdkit.Chem.Descriptors import CalcMolDescriptors
import numpy as np
'''
Considerations: 
- Not all descriptors available for a given molecule
- Need to ensure consistency across all datasets
'''
desc_list = []
for mol in valid_compounds["rdkit_mol"]:
    desc = CalcMolDescriptors(mol, missingVal=np.nan)
    desc_list.append(desc)
   

# Convert list of descriptors to a DataFrame with descriptor names as columns
desc_df = pd.DataFrame(desc_list)
print(desc_df.head())
print(f"Shape before dropping: {desc_df.shape}")
print(f"Columns with NaN: {desc_df.isna().any().sum()}")
print([c for c in desc_df.columns if c in ['MolWt', 'MolLogP', 'TPSA', 'ExactMolWt']])
desc_df = desc_df.dropna(axis=1)
print(f"Shape after dropping: {desc_df.shape}")
X = desc_df.drop(columns=["MolWt", "ExactMolWt", "TPSA", "MolLogP"]).values
y = desc_df[["MolWt", "TPSA", "MolLogP"]].values
print(f"X shape: {X.shape}, y shape: {y.shape}")
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Second split: Split temp into 2: 0.5 x 20% temp = 10% test + 10% val
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
# Normalize input features using training set statistics

# Step 1: Compute std from training set
std_input = X_train.std(axis=0)
print(f"Std shape: {std_input.shape}")

# Avoid descriptors with zero variance (std=0) to prevent division by zero
zero_std_mask = std_input == 0
print(f"Number of descriptors with zero std: {zero_std_mask.sum()}")

# Drop zero variance descriptors from all datasets
X_train_non_zero_var = X_train[:, ~zero_std_mask]
X_val_non_zero_var = X_val[:, ~zero_std_mask]
X_test_non_zero_var = X_test[:, ~zero_std_mask]

mean_input_non_zero_var = X_train_non_zero_var.mean(axis=0)
std_input_non_zero_var = X_train_non_zero_var.std(axis=0)
print(f"Mean shape after dropping zero variance: {mean_input_non_zero_var.shape}, Std shape after dropping zero variance: {std_input_non_zero_var.shape}")

# Step 2: Normalize training, validation, and test sets using training set mean and std
X_train_norm = (X_train_non_zero_var - mean_input_non_zero_var) / std_input_non_zero_var
X_val_norm = (X_val_non_zero_var - mean_input_non_zero_var) / std_input_non_zero_var
X_test_norm = (X_test_non_zero_var - mean_input_non_zero_var) / std_input_non_zero_var

print(f"Train mean (should be ~0): {X_train_norm.mean(axis=0).mean():.4f}")
print(f"Train std (should be ~1): {X_train_norm.std(axis=0).mean():.4f}")
print(f"dtype: {X_train_norm.dtype}")

# Normalize target using training set statistics

# Step 1: Compute std from training set
std_target = y_train.std(axis=0)
print(f"Std shape: {std_target.shape}")

# Avoid descriptors with zero variance (std=0) to prevent division by zero
zero_std_mask = std_target == 0
print(f"Number of targets with zero std: {zero_std_mask.sum()}")

# Drop zero variance descriptors from all datasets
y_train_non_zero_var = y_train[:, ~zero_std_mask]
y_val_non_zero_var = y_val[:, ~zero_std_mask]
y_test_non_zero_var = y_test[:, ~zero_std_mask]

mean_target_non_zero_var = y_train_non_zero_var.mean(axis=0)
std_target_non_zero_var = y_train_non_zero_var.std(axis=0)
print(f"Mean shape after dropping zero variance: {mean_target_non_zero_var.shape}, Std shape after dropping zero variance: {std_target_non_zero_var.shape}")

# Step 2: Normalize training, validation, and test sets using training set mean and std
y_train_norm = (y_train_non_zero_var - mean_target_non_zero_var) / std_target_non_zero_var
y_val_norm = (y_val_non_zero_var - mean_target_non_zero_var) / std_target_non_zero_var
y_test_norm = (y_test_non_zero_var - mean_target_non_zero_var) / std_target_non_zero_var

print(f"Train mean (should be ~0): {y_train_norm.mean(axis=0).mean():.4f}")
print(f"Train std (should be ~1): {y_train_norm.std(axis=0).mean():.4f}")
print(f"dtype: {y_train_norm.dtype}")

# Convert Datasets to PyTorch tensors
import torch

X_train_tensor = torch.tensor(X_train_norm, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_norm, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_norm, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_norm, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_norm, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_norm, dtype=torch.float32)

print(f"X_train: {X_train_tensor.shape}, y_train: {y_train_tensor.shape}")

from torch.utils.data import Dataset, DataLoader

class MoleculeDataset(Dataset):
    def __init__(self, descriptors, labels):
        self.descriptors = descriptors
        self.labels = labels
    
    def __len__(self):
        return len(self.descriptors)
    
    def __getitem__(self, idx):
        return self.descriptors[idx], self.labels[idx]

# Create dataset and dataloader
X_train_dataset = MoleculeDataset(X_train_tensor, y_train_tensor)
X_val_dataset = MoleculeDataset(X_val_tensor, y_val_tensor)
X_test_dataset = MoleculeDataset(X_test_tensor, y_test_tensor)

X_train_loader = DataLoader(X_train_dataset, batch_size=32, shuffle=True)
X_val_loader = DataLoader(X_val_dataset, batch_size=32, shuffle=False)
X_test_loader = DataLoader(X_test_dataset, batch_size=32, shuffle=False)



Number of compounds with invalid SMILES: 0
   MaxAbsEStateIndex  MaxEStateIndex  MinAbsEStateIndex  MinEStateIndex  \
0          10.253329       10.253329           0.486602       -1.701605   
1          11.724911       11.724911           0.145880       -0.145880   
2          10.020498       10.020498           0.845090        0.845090   
3           2.270278        2.270278           1.301055        1.301055   
4           2.041667        2.041667           1.712963        1.712963   

        qed        SPS    MolWt  HeavyAtomMolWt  ExactMolWt  \
0  0.217518  41.062500  457.432         430.216  457.158411   
1  0.811283   9.933333  201.225         190.137  201.078979   
2  0.343706  11.000000  152.237         136.109  152.120115   
3  0.291526  11.636364  278.354         264.242  278.109550   
4  0.448927   8.000000   84.143          80.111   84.003371   

   NumValenceElectrons  ...  fr_sulfide  fr_sulfonamd  fr_sulfone  \
0                  178  ...           0             0     

In [4]:
# Model
import torch.nn as nn

full_model = nn.Sequential(
    # --- Block 1 ---
    # Projects 194 raw input features (data, not neurons) into 512 neurons
    nn.Linear(194, 512),
    nn.BatchNorm1d(512),  # normalises the 512 neuron outputs
    nn.ReLU(),            # introduces non-linearity
    nn.Dropout(0.2),      # randomly deactivates 20% of neurons during training

    # --- Block 2 ---
    # Hidden layer: 512 neurons → 256 neurons
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.2),

    # --- Block 3 ---
    # Hidden layer: 256 neurons → 128 neurons
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.2),

    # --- Output ---
    # No activation, BatchNorm, or Dropout — raw value for regression
    nn.Linear(128, 3)
)


# loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(full_model.parameters(), lr=0.001)



In [5]:
# Use patience of 10 for early stopping and patience of 5 for the scheduler
torch.manual_seed(42) # For reproducibility
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,      # halve the learning rate
    patience=5,      # after 5 epochs with no improvement
    min_lr=1e-6      # never go below this
)

num_epochs = 100
best_val_loss = float('inf')
early_stopping_patience = 10
patience_counter = 0

for epoch in range(num_epochs):
    # --- Training loop ---
    full_model.train() # Set model to training mode (enables dropout, batchnorm updates)
    train_loss = 0.0
    for batch in X_train_loader:
        X_batch, y_batch = batch
        optimizer.zero_grad()           # Clear gradients
        outputs = full_model(X_batch)   # Forward pass
        loss = criterion(outputs, y_batch) # Compute loss
        loss.backward()                 # Backpropagation
        optimizer.step()                # Update weights
        train_loss += loss.item() # Accumulate loss
    
    # --- Validation loop ---
    full_model.eval() # Set model to evaluation mode (disables dropout, batchnorm updates
    val_loss = 0.0
    with torch.no_grad(): # No need to compute gradients during validation
        for batch in X_val_loader:
            X_batch, y_batch = batch
            outputs = full_model(X_batch)   # Forward pass
            loss = criterion(outputs, y_batch) # Compute loss
            val_loss += loss.item() # Accumulate loss
    
    # --- Early stopping check ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(full_model.state_dict(), 'best_model.pt')  # ← save here
        print(f"Epoch {epoch}/{num_epochs}, Train Loss: {train_loss/len(X_train_loader):.4f}, Val Loss: {val_loss/len(X_val_loader):.4f}")
    else:
        # weights here do not reflect that of the best epoch
        # That's why we save the weights at the best epoch, not here
        patience_counter += 1
        if patience_counter >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch}")
            print(f"Epoch {epoch}/{num_epochs}, Train Loss: {train_loss/len(X_train_loader):.4f}, Val Loss: {val_loss/len(X_val_loader):.4f}")
            break  # weights already saved at best epoch

    scheduler.step(val_loss / len(X_val_loader))  # ← after the if/else, once per epoch
    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{num_epochs}, Train Loss: {train_loss/len(X_train_loader):.4f}, Val Loss: {val_loss/len(X_val_loader):.4f}")

# Restore best weights after training
full_model.load_state_dict(torch.load('best_model.pt'))



Epoch 0/100, Train Loss: 0.3103, Val Loss: 0.1375
Epoch 0/100, Train Loss: 0.3103, Val Loss: 0.1375
Epoch 1/100, Train Loss: 0.1708, Val Loss: 0.0766
Epoch 2/100, Train Loss: 0.1378, Val Loss: 0.0401
Epoch 5/100, Train Loss: 0.1306, Val Loss: 0.0399
Epoch 7/100, Train Loss: 0.1080, Val Loss: 0.0337
Epoch 10/100, Train Loss: 0.1025, Val Loss: 0.0318
Epoch 10/100, Train Loss: 0.1025, Val Loss: 0.0318
Epoch 11/100, Train Loss: 0.0898, Val Loss: 0.0266
Epoch 17/100, Train Loss: 0.0734, Val Loss: 0.0229
Epoch 19/100, Train Loss: 0.0897, Val Loss: 0.0217
Epoch 20/100, Train Loss: 0.0854, Val Loss: 0.0216
Epoch 20/100, Train Loss: 0.0854, Val Loss: 0.0216
Epoch 24/100, Train Loss: 0.0740, Val Loss: 0.0174
Epoch 30/100, Train Loss: 0.0874, Val Loss: 0.0317
Epoch 32/100, Train Loss: 0.0776, Val Loss: 0.0168
Epoch 36/100, Train Loss: 0.0832, Val Loss: 0.0147
Epoch 40/100, Train Loss: 0.0870, Val Loss: 0.0298
Early stopping at epoch 46
Epoch 46/100, Train Loss: 0.0631, Val Loss: 0.0226


<All keys matched successfully>